# RAG Benchmark — Generation Stage on Colab

Second run. Three things changed since the first, and each is a fix to a
limitation rather than a repeat:

| Change | Why |
|---|---|
| Judge is **phi3**, not mistral | mistral was one of the models it was judging, and came out top of the metric it controlled |
| Faithfulness sample is **shared across models** | the old sample overlapped ~21%, forcing an unpaired test; it is now 100% |
| CaseHOLD and SciQ supply their **options** | both were answered open-ended, so neither had a correctness measure |

The previous run is left in the database untouched, as a record. This produces a
new run id, and everything downstream is scored and tested against that id.

**Retrieval is not repeated.** Corpora and indexes stay valid, so the 12-hour
retrieval benchmark is not re-run.

Expect roughly **12–18 hours**: about 5 for generation and the rest for
faithfulness, which costs two judge calls per answer. Run the cells in order.

## 1 · Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

# 7-9B models at fp16 need roughly 18 GB. On a 16 GB card Ollama will fall back
# to partial CPU offload, which still works but is several times slower.

## 1b · Keep the session alive

Colab disconnects a browser it considers idle after roughly ninety minutes,
**regardless of what is still computing**. That, not the twelve-hour cap, is
what ends most long unattended runs: the GPU is working, nobody is clicking,
and the session is dropped anyway.

Run the cell below, then also paste the printed snippet into the browser
console (**F12 → Console**). The console copy is the one that reliably works,
because a notebook output frame is sandboxed away from the page chrome that
owns the connect button.

Neither survives a **closed tab or a sleeping machine**. Leave the tab open,
the lid up, and the laptop on mains power.

In [ ]:
KEEPALIVE_JS = """
setInterval(() => {
  const b = document.querySelector("colab-connect-button");
  if (b && b.shadowRoot) {
    b.shadowRoot.querySelector("#connect")?.click();
    console.log("ping", new Date().toLocaleTimeString());
  }
}, 60000);
"""

try:
    from google.colab import output
    output.eval_js(KEEPALIVE_JS)
    print('keep-alive registered from the notebook (best effort)\n')
except Exception as exc:
    print(f'could not register from the notebook: {exc}\n')

print('Paste this into the browser console (F12 -> Console) as well:')
print('-' * 70)
print(KEEPALIVE_JS.strip())
print('-' * 70)
print('You should then see a "ping" line every minute in the console.')
print('If you do not, it is not running and the session will idle out.')

## 2 · Mount Drive and verify the layout

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive/AI Benchmark')
DRIVE_DATASETS = DRIVE / 'datasets'
DRIVE_ARTIFACTS = DRIVE / 'artifacts'

print(f'Drive folder: {DRIVE}')
if not DRIVE.exists():
    raise SystemExit(
        f'{DRIVE} not found. Check the folder name is exactly "AI Benchmark" '
        'and that it sits at the top level of My Drive.'
    )

required = {
    'datasets/':                DRIVE_DATASETS,
    'artifacts/corpora/':       DRIVE_ARTIFACTS / 'corpora',
    'artifacts/benchmark.sqlite': DRIVE_ARTIFACTS / 'benchmark.sqlite',
}
missing = []
for label, path in required.items():
    ok = path.exists()
    print(f'  {"OK  " if ok else "MISS"}  {label}')
    if not ok:
        missing.append(label)
if missing:
    raise SystemExit(
        'Missing from Drive: ' + ', '.join(missing) +
        '\nUpload them from C:\\AI BENCHMARK\\ before continuing.'
    )

## 3 · Clone the repository

In [ ]:
import subprocess
from pathlib import Path

REPO = 'https://github.com/mahesh062003/ai-benchmark.git'
PROJECT = Path('/content/ai-benchmark')

if PROJECT.exists():
    print('updating existing clone')
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO, str(PROJECT)], check=True)

%cd /content/ai-benchmark
!git log --oneline -1

## 4 · Install Python dependencies

In [ ]:
# Colab already ships torch, numpy and pandas; pip will leave those alone.
!pip install -q -r /content/ai-benchmark/requirements.txt 2>&1 | tail -5
print('dependencies ready')

## 5 · Install Ollama and pull the four models and the judge

About **19 GB** of model weights. They are stored on Colab's local disk, not
Drive, because loading a model over FUSE is far slower than re-downloading it.
Expect 10–20 minutes on the first run of each session.

In [ ]:
import os
import subprocess
import time

import requests

os.environ['OLLAMA_MODELS'] = '/content/ollama_models'
os.makedirs('/content/ollama_models', exist_ok=True)

if subprocess.run(['which', 'ollama'], capture_output=True).returncode != 0:
    # Ollama now ships its release as .tar.zst. Colab has no zstd binary, so the
    # official installer fails at the extraction step and exits 1.
    print('installing zstd, then ollama...')
    subprocess.run('apt-get -qq update && apt-get -qq install -y zstd', shell=True)
    if subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True).returncode != 0:
        print('installer failed; falling back to the release binary')
        subprocess.run(
            'curl -L --retry 3 -o /tmp/ollama.tar.zst '
            'https://github.com/ollama/ollama/releases/latest/download/ollama-linux-amd64.tar.zst',
            shell=True, check=True,
        )
        subprocess.run(
            'tar --use-compress-program=unzstd -xf /tmp/ollama.tar.zst -C /usr/local',
            shell=True, check=True,
        )
else:
    print('ollama already installed')

# Start the server detached; it must outlive this cell.
subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    env={**os.environ},
)

for attempt in range(60):
    try:
        if requests.get('http://localhost:11434/api/tags', timeout=2).ok:
            print('ollama server is up')
            break
    except Exception:
        pass
    time.sleep(2)
else:
    raise SystemExit('ollama did not start; re-run this cell')

MODELS = ['llama3.1', 'gemma2', 'mistral', 'qwen2.5']

# The judge is deliberately not one of the models under test. In the first run
# mistral judged its own answers and scored highest on the metric it was
# judging, which cannot be resolved from inside the experiment. phi3 is outside
# the compared set, and smaller than all four, which matters because
# faithfulness costs two judge calls per answer.
JUDGE = 'phi3'

# Pulls fail intermittently -- a dropped connection to the registry is enough.
# Left unhandled that aborts the cell midway through a 21 GB download and the
# whole step has to be repeated by hand. Already-pulled layers are cached, so a
# retry resumes rather than starting over.
!df -h /content | tail -1

def pull(model, attempts=4):
    for attempt in range(1, attempts + 1):
        result = subprocess.run(['ollama', 'pull', model],
                                capture_output=True, text=True)
        if result.returncode == 0:
            return
        tail = (result.stderr or result.stdout or '').strip().splitlines()
        print(f'  attempt {attempt}/{attempts} failed: {tail[-1] if tail else "no output"}')
        if attempt < attempts:
            time.sleep(10 * attempt)
    raise SystemExit(
        f'could not pull {model} after {attempts} attempts. Check the disk space '
        'printed above, then re-run this cell -- cached layers are kept.'
    )

for model in MODELS + [JUDGE]:
    print(f'--- pulling {model} ---')
    pull(model)

!ollama list

## 6 · Stage artifacts onto local disk

Copied from Drive so the database is written on a real filesystem. Only the
corpora and the database are needed; the FAISS indexes are not.

In [ ]:
import shutil
from pathlib import Path

LOCAL_ARTIFACTS = Path('/content/artifacts')
LOCAL_ARTIFACTS.mkdir(parents=True, exist_ok=True)
(LOCAL_ARTIFACTS / 'results').mkdir(exist_ok=True)

local_db = LOCAL_ARTIFACTS / 'benchmark.sqlite'
local_corpora = LOCAL_ARTIFACTS / 'corpora'

if not local_corpora.exists():
    print('copying corpora from Drive (a few minutes)...')
    shutil.copytree(DRIVE_ARTIFACTS / 'corpora', local_corpora)
    print('  done')
else:
    print('corpora already staged')

# Always take the newest database so a resumed session continues the same run.
print('copying database from Drive...')
shutil.copy2(DRIVE_ARTIFACTS / 'benchmark.sqlite', local_db)

# Restore the frozen task sets. Without these a resumed session re-runs
# retrieval to rebuild them, which costs about fifteen minutes on MedQA alone.
drive_results = DRIVE_ARTIFACTS / 'results'
restored = 0
if drive_results.exists():
    for item in drive_results.glob('generation_tasks_*.json'):
        shutil.copy2(item, LOCAL_ARTIFACTS / 'results' / item.name)
        restored += 1
print(f'restored {restored} frozen task set(s)')

import sqlite3
with sqlite3.connect(f'file:{local_db}?mode=ro', uri=True) as c:
    runs = c.execute('SELECT run_id FROM runs ORDER BY created_at DESC').fetchall()
    agg = c.execute('SELECT COUNT(*) FROM aggregate_metrics').fetchone()[0]
    gens = c.execute('SELECT COUNT(*) FROM generations').fetchone()[0]
print(f'  runs={[r[0] for r in runs]}  aggregate_metrics={agg}  generations={gens}')
if gens:
    print(f'  resuming: {gens} answers already generated will be skipped')

## 7 · Point the framework at these directories

In [ ]:
import os

# Datasets stay on Drive: they are read rarely and never written.
os.environ['RAGBENCH_DATASETS_DIR'] = str(DRIVE_DATASETS)
# Artifacts live on local disk while running, and are synced back to Drive.
os.environ['RAGBENCH_ARTIFACTS_DIR'] = str(LOCAL_ARTIFACTS)

!python -m cli datasets 2>&1 | head -12
print()
print('datasets dir :', os.environ['RAGBENCH_DATASETS_DIR'])
print('artifacts dir:', os.environ['RAGBENCH_ARTIFACTS_DIR'])

## 8 · Start the automatic Drive sync

Copies the database to Drive every 10 minutes using SQLite's online backup,
which is safe while the database is being written. Leave this running.

In [ ]:
import sqlite3
import threading
import time
from datetime import datetime

SYNC_SECONDS = 600
_stop_sync = threading.Event()


def sync_to_drive(reason='periodic'):
    # Back the live database up to Drive without interrupting writers, and carry
    # the frozen task sets across so a resumed session does not rebuild them.
    try:
        source = sqlite3.connect(f'file:{local_db}?mode=ro', uri=True)
        target = sqlite3.connect(str(DRIVE_ARTIFACTS / 'benchmark.sqlite'))
        with target:
            source.backup(target)
        source.close()
        target.close()

        import shutil
        drive_results = DRIVE_ARTIFACTS / 'results'
        drive_results.mkdir(parents=True, exist_ok=True)
        for item in (LOCAL_ARTIFACTS / 'results').glob('generation_tasks_*.json'):
            shutil.copy2(item, drive_results / item.name)

        stamp = datetime.now().strftime('%H:%M:%S')
        print(f'[{stamp}] synced to Drive ({reason})')
    except Exception as exc:                     # never kill the run over a sync
        print(f'sync failed ({exc}); the local database is still intact')


def _loop():
    while not _stop_sync.wait(SYNC_SECONDS):
        sync_to_drive()


threading.Thread(target=_loop, daemon=True).start()
print(f'auto-sync every {SYNC_SECONDS // 60} minutes -> {DRIVE_ARTIFACTS / "benchmark.sqlite"}')

## 8b · Give CaseHOLD and SciQ their options back

Both are multiple-choice datasets, but the first run asked them open-ended, so
`choice_correct` is NULL for all 2,400 of their answers. The loaders now build
the option sets — but the metadata is cached in **two** places, and both have to
go:

| Cache | Cleared by |
|---|---|
| `queries.jsonl` beside each corpus | `refresh-query-metadata` rewrites it |
| `generation_tasks_*.json` — embeds its own copy | `refresh-query-metadata` deletes it, and the cell below clears Drive's copies |

The second one is the trap. A frozen task set carries a snapshot of every
query's metadata, so a run that loads a stale one uses the old metadata and
looks completely normal for five hours before producing NULL again.

Rebuilding the task set costs a retrieval pass over the sampled queries, about
fifteen to twenty minutes, most of it MedQA.

In [ ]:
# Inspect the frozen task sets BEFORE anything deletes them. Whether they are
# stale decides whether the run id can still be resumed: if the task set was
# built before the loader fix then the answers already stored under that run id
# came from option-less prompts, and resuming would silently keep them.
import glob
import json as _json
import os

MULTIPLE_CHOICE = ('casehold', 'sciq', 'medqa')


def is_stale(path):
    data = _json.loads(open(path, encoding='utf-8').read())
    seen = {}
    for task in data.get('tasks', []):
        name = task['dataset']
        if name in MULTIPLE_CHOICE and name not in seen:
            seen[name] = bool((task.get('metadata') or {}).get('options'))
    return [name for name, ok in sorted(seen.items()) if not ok]


existing = glob.glob(f'{LOCAL_ARTIFACTS}/results/generation_tasks_*.json')
existing += [str(p) for p in (DRIVE_ARTIFACTS / 'results').glob('generation_tasks_*.json')]
stale_for = sorted({name for path in existing for name in is_stale(path)})
if existing:
    print(f'found {len(existing)} frozen task set(s); stale for: {stale_for or "nothing"}')

!python -m cli refresh-query-metadata --datasets casehold,sciq --config config/default.yaml

# The command clears task sets on local disk. Drive keeps its own copies and the
# staging cell restores them, so they have to go as well or the next session
# quietly reintroduces the stale metadata.
for item in list((DRIVE_ARTIFACTS / 'results').glob('generation_tasks_*.json')):
    item.unlink()
    print(f'removed stale task set from Drive: {item.name}')
for item in list((LOCAL_ARTIFACTS / 'results').glob('generation_tasks_*.json')):
    item.unlink()
    print(f'removed stale task set locally: {item.name}')

# Only discard the resume key when the task set really was stale. A routine
# reconnect must keep it -- clearing it there would throw away every answer
# generated so far and restart a multi-hour run from zero.
RUN_FILE = DRIVE_ARTIFACTS / 'generation_run_id_v2.txt'
if stale_for and RUN_FILE.exists():
    print(f'
discarding run id {RUN_FILE.read_text().strip()}: its answers were '
          f'generated from prompts with no options for {", ".join(stale_for)}')
    RUN_FILE.unlink()
elif RUN_FILE.exists():
    print(f'
keeping run id {RUN_FILE.read_text().strip()} -- it is resumable')

# Prove the options actually landed before spending GPU time on generation.
for name in ('casehold', 'sciq'):
    path = glob.glob(f'{LOCAL_ARTIFACTS}/corpora/{name}/*/*/queries.jsonl')[0]
    row = _json.loads(open(path, encoding='utf-8').readline())
    options = row['metadata'].get('options', {})
    gold = row['metadata'].get('answer_idx')
    ok = bool(options) and options.get(gold) == row['answer']
    print(f'{name:9s} options={len(options)} gold={gold!r} correct_answer_matches={ok}')
    if not ok:
        raise SystemExit(f'{name} has no usable options -- do not generate yet')

## 9 · Generate

100 questions per dataset x 6 datasets x 3 strategies x 4 models = **7,200
answers**. Models run one at a time to avoid reloading, and every answer is
committed as it is produced, so an interrupted run resumes rather than
restarting.

In [ ]:
import glob
import json as _json
import os
import re
import sqlite3 as _sq
import subprocess

# A separate file from the first run's, so this starts a genuinely new run
# instead of resuming the old one. Within this run, resume still works exactly
# as before: the id is written to Drive the moment it appears, so a session that
# dies minutes later can be continued rather than restarted.
RUN_FILE = DRIVE_ARTIFACTS / 'generation_run_id_v2.txt'
EXPECTED = 7200          # 100 queries x 6 datasets x 3 strategies x 4 models

MULTIPLE_CHOICE = ('casehold', 'sciq', 'medqa')


def task_set_problem():
    """Why the frozen task set is unusable, or None if it is fine.

    The task set embeds a copy of each query's metadata, so a stale one sends
    the multiple-choice datasets through as open-ended questions. That failure
    is invisible until the run finishes, which is why it is checked here rather
    than trusted.
    """
    files = glob.glob(f'{LOCAL_ARTIFACTS}/results/generation_tasks_*.json')
    if not files:
        return None                      # nothing frozen yet; it will be built
    newest = max(files, key=os.path.getmtime)
    data = _json.loads(open(newest, encoding='utf-8').read())
    seen = {}
    for task in data.get('tasks', []):
        dataset = task['dataset']
        if dataset in MULTIPLE_CHOICE and dataset not in seen:
            seen[dataset] = bool((task.get('metadata') or {}).get('options'))
    missing = sorted(name for name, ok in seen.items() if not ok)
    if missing:
        return (
            f'the frozen task set {os.path.basename(newest)} carries no options for '
            f'{", ".join(missing)}. It predates the loader fix. Delete every '
            'generation_tasks_*.json from /content/artifacts/results and from '
            'Drive, then re-run section 8b and this cell.'
        )
    return None


problem = task_set_problem()
if problem:
    raise SystemExit(problem)

command = ['python', '-m', 'cli', 'generate-all', '--all',
           '--models', 'llama3.1,gemma2,mistral,qwen2.5', '--limit', '100',
           '--config', 'config/default.yaml', '--verbose']

if RUN_FILE.exists():
    run_id = RUN_FILE.read_text().strip()
    # Count answers in THIS run only. Counting every generation would include
    # the first run's 7,200 and skip the stage before it had produced anything.
    with _sq.connect(f'file:{local_db}?mode=ro', uri=True) as _c:
        done = _c.execute(
            'SELECT COUNT(*) FROM generations WHERE run_id = ? AND answer IS NOT NULL',
            (run_id,),
        ).fetchone()[0]
    if done >= EXPECTED:
        print(f'generation already complete for {run_id} ({done} answers) -- skip to scoring')
        raise SystemExit(0)
    print(f'RESUMING run {run_id} -- {done} of {EXPECTED} already stored\n')
    command += ['--run', run_id]
else:
    print('starting a NEW generation run\n')

process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1)
checked = False
for line in process.stdout:
    print(line, end='')

    if not RUN_FILE.exists():
        found = re.search(r'(genall-[0-9a-f]+)', line)
        if found:
            RUN_FILE.write_text(found.group(1))
            print(f'\n>>> run id saved to Drive: {found.group(1)}\n')

    # The moment the task set exists, check it and stop before the long stretch
    # rather than after it.
    if not checked and ('froze' in line or 'frozen generation tasks' in line):
        checked = True
        problem = task_set_problem()
        if problem:
            process.terminate()
            raise SystemExit(f'STOPPED: {problem}')
        print('\n>>> task set carries the multiple-choice options — proceeding\n')

process.wait()

In [ ]:
sync_to_drive('after generation')

import sqlite3
with sqlite3.connect(f'file:{local_db}?mode=ro', uri=True) as c:
    print('answers generated:', c.execute('SELECT COUNT(*) FROM generations').fetchone()[0])
    for row in c.execute(
        'SELECT model, COUNT(*), SUM(answer IS NULL) FROM generations GROUP BY model'
    ):
        print(f'  {row[0]:12s} {row[1]:6d} answers, {row[2] or 0} empty')

## 10 · Score the answers

RAGAS faithfulness with a fixed judge (`mistral`, set in `config/default.yaml`)
so every model is rated by the same rater, plus NLI hallucination detection.
Runs after generation and never during it.

In [ ]:
import subprocess
import time

import requests

# The Ollama server can die between generation and scoring. Left unchecked,
# score-answers logs "skipping faithfulness" and exits successfully, so the run
# looks finished while half the measurement is missing. Restart it first.
def ollama_ready(timeout_seconds=120):
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        try:
            if requests.get('http://localhost:11434/api/tags', timeout=2).ok:
                return True
        except Exception:
            pass
        time.sleep(2)
    return False


if not ollama_ready(timeout_seconds=4):
    print('ollama is down; restarting it before scoring')
    subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL,
                     stderr=subprocess.DEVNULL, env={**os.environ})
    if not ollama_ready():
        raise SystemExit('ollama would not start -- re-run this cell')
print('ollama is up; starting scoring\n')

In [ ]:
# --run is what keeps this honest. Without it, score-answers would draw its
# faithfulness sample from every unscored row in the database, which now
# includes the first run's answers, and the 1,440 judgements would be split
# across two runs with two different judges.
#
# --config is not optional either. Without it the CLI falls back to dataclass
# defaults, where faithfulness_sample is None -- judging all 7,200 answers
# rather than the shared 1,440 -- and the judge is the single-model default
# rather than phi3. The run still looks correct while doing five times the work
# with the wrong rater.
#
# Run with ! rather than subprocess so the progress bars stream into the cell.
# A silent long-running job is indistinguishable from a hung one.
RUN_ID = RUN_FILE.read_text().strip()
print(f'scoring run {RUN_ID}\n')

!python -m cli score-answers --run {RUN_ID} --config config/default.yaml --verbose

In [ ]:
# Confirm both halves actually ran, and that the sample really is paired --
# an unpaired sample would silently cost the paired test again.
import itertools
import sqlite3

with sqlite3.connect(f'file:{local_db}?mode=ro', uri=True) as c:
    c.row_factory = sqlite3.Row
    faith = c.execute(
        'SELECT COUNT(*) FROM generations WHERE run_id = ? AND faithfulness IS NOT NULL',
        (RUN_ID,)).fetchone()[0]
    halluc = c.execute(
        'SELECT COUNT(*) FROM generations WHERE run_id = ? AND hallucination IS NOT NULL',
        (RUN_ID,)).fetchone()[0]
    judges = [r[0] for r in c.execute(
        'SELECT DISTINCT faithfulness_judge FROM generations WHERE run_id = ?'
        ' AND faithfulness IS NOT NULL', (RUN_ID,))]
    rows = c.execute(
        'SELECT dataset, method, model, query_id FROM generations'
        ' WHERE run_id = ? AND faithfulness IS NOT NULL', (RUN_ID,)).fetchall()

print(f'faithfulness scored : {faith}')
print(f'hallucination scored: {halluc}')
print(f'judge               : {judges}')

by_model = {}
for row in rows:
    by_model.setdefault(row['model'], set()).add(
        (row['dataset'], row['method'], row['query_id']))
if len(by_model) > 1:
    overlaps = [
        len(by_model[a] & by_model[b]) / max(1, min(len(by_model[a]), len(by_model[b])))
        for a, b in itertools.combinations(sorted(by_model), 2)
    ]
    print(f'model pair overlap  : {min(overlaps):.0%} (100% means the tests are paired)')

sync_to_drive('after scoring')

if faith == 0:
    raise SystemExit(
        'FAITHFULNESS DID NOT RUN. Check the ollama messages above and re-run '
        'the scoring cell before continuing -- do not run the final cell yet, '
        'it stops the Drive backup.'
    )

In [ ]:
!python -m cli results --config config/default.yaml

## 11 · Final sync and export

Run this only once scoring has finished. It stops the periodic Drive backup, so
any work done after it is unprotected until the next manual sync.

In [ ]:
import shutil
import sqlite3
import subprocess

# Refuse to stop the backup while a measurement is still missing: that
# combination is what loses a long unattended run.
with sqlite3.connect(f'file:{local_db}?mode=ro', uri=True) as c:
    faith = c.execute(
        'SELECT COUNT(*) FROM generations WHERE faithfulness IS NOT NULL').fetchone()[0]
if faith == 0:
    raise SystemExit(
        'Faithfulness has not been scored. Re-run the scoring cell first; '
        'stopping the Drive backup now would leave a later run unprotected.'
    )

_stop_sync.set()
sync_to_drive('final')

# Results CSVs are small; copy the whole results directory across.
drive_results = DRIVE_ARTIFACTS / 'results'
drive_results.mkdir(parents=True, exist_ok=True)
for item in (LOCAL_ARTIFACTS / 'results').glob('*'):
    if item.is_file():
        shutil.copy2(item, drive_results / item.name)
        print('copied', item.name)

!python -m cli export --config config/default.yaml

# Significance tests retrieval, so it needs the retrieval run. Left to itself it
# picks the most recent run, which by now is a generation run carrying no
# per-query retrieval metrics, and reports "nothing to test".
with sqlite3.connect(f'file:{local_db}?mode=ro', uri=True) as c:
    retrieval_run = c.execute(
        "SELECT run_id FROM runs WHERE stage LIKE 'retrieval%'"
        " ORDER BY created_at DESC LIMIT 1"
    ).fetchone()

if retrieval_run:
    print(f'\ntesting significance on retrieval run {retrieval_run[0]}')
    subprocess.run(
        ['python', '-m', 'cli', 'significance',
         '--config', 'config/default.yaml', '--run', retrieval_run[0]],
        check=False,
    )
else:
    print('no retrieval run found; skipping significance')

for item in (LOCAL_ARTIFACTS / 'results').glob('*.csv'):
    shutil.copy2(item, drive_results / item.name)

print()
print('Everything is on Drive. Download artifacts/benchmark.sqlite to your')
print('laptop, drop it into C:\\AI BENCHMARK\\artifacts\\, and run:')
print('    streamlit run dashboard/app.py')

# Both halves of the study get significance testing. The retrieval command
# picks the latest run that has retrieval metrics; the generation one is
# pinned to this run explicitly.
!python -m cli export --config config/default.yaml
!python -m cli significance --config config/default.yaml
!python -m cli significance-generation --run {RUN_ID} --config config/default.yaml

import shutil as _sh
for name in ('aggregates.csv', 'significance.csv', 'significance_generation.csv'):
    source = Path(local_artifacts) / 'results' / name
    if source.exists():
        _sh.copy(source, DRIVE_ARTIFACTS / name)
        print(f'copied {name}')

sync_to_drive('final')


## Resuming after a disconnect

Re-run the cells in order. Everything is idempotent:

- **Sections 1–8** rebuild the environment; staging skips files already present.
- **Section 8b** reports the options are already there and changes nothing.
- **Section 9** reads the run id from Drive and skips every answer already
  stored, so a reconnect costs minutes rather than hours.
- **Section 10** scores only rows that are still NULL.

Disconnects here have always been the ~90 minute idle timer, not the 12 hour
cap, so run section 1b and leave the tab open.

### If you need to re-judge from scratch

`score-answers` only fills rows that are NULL, so pointing it at a different
judge would silently score nothing. Clear the column first:

```
!python -m cli reset-scores --column faithfulness --run {RUN_ID} --yes --config config/default.yaml
```

It refuses without `--yes`, reports how many scores it discarded, and never
touches the answers themselves.